In [1]:
!pip install pandas matplotlib openpyxl

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 28.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 36.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [matplotlib]8 [matplotlib]


In [4]:
import pandas as pd

# Updated to use your new Desktop path
file_path = "/Users/ayman/Desktop/cleaned_dashboard_data.csv"

df = pd.read_csv(file_path)
df.head()

,CustomerId,Gender,Marital Status,Number of Dependents,Occupation,Income,Education Level,Customer Tenure,Customer Segment,Preferred Communication Channel,Credit Score,Credit History Length,Outstanding Loans,Churn Flag,Churn Reason,Churn Date,Balance,NumOfProducts,NumComplaints,Age
0,83ef0b54-35f6-4f84-af58-5653ac0c0dc4,Male,Divorced,3,Information systems manager,77710.14,High School,30,Retail,Phone,397,24,41959.74,0,NaN,NaN,211359.05,1,0,38
1,009f115a-e5ca-4cf4-97d6-530140545e4e,Female,Married,1,Charity fundraiser,58209.87,High School,27,SME,Email,665,10,8916.67,0,NaN,NaN,30624.76,4,1,26
2,66309fd3-5009-44d3-a3f7-1657c869d573,Female,Single,1,Television production assistant,9794.01,High School,14,Retail,Email,715,21,43270.54,0,NaN,NaN,111956.61,2,6,72
3,b02a30df-1a5f-4087-8075-2a35432da641,Female,Divorced,5,Agricultural engineer,15088.98,High School,23,Corporate,Phone,747,17,17887.65,0,NaN,NaN,201187.61,1,0,35
4,0d932e5b-bb3a-4104-8c83-f84270f7f2ea,Female,Divorced,2,"Teacher, early years/pre",60726.56,Master's,22,Corporate,Email,549,25,32686.84,0,NaN,NaN,60391.24,5,6,34


In [5]:
# Run the quality checks
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Value Counts:\n{df['Churn Flag'].value_counts()}")
print(f"Churn Rate: {df['Churn Flag'].mean() * 100:.2f}%")

Shape: (115640, 20)
Columns: ['CustomerId', 'Gender', 'Marital Status', 'Number of Dependents', 'Occupation', 'Income', 'Education Level', 'Customer Tenure', 'Customer Segment', 'Preferred Communication Channel', 'Credit Score', 'Credit History Length', 'Outstanding Loans', 'Churn Flag', 'Churn Reason', 'Churn Date', 'Balance', 'NumOfProducts', 'NumComplaints', 'Age']
Value Counts:
Churn Flag
0    101546
1     14094
Name: count, dtype: int64
Churn Rate: 12.19%


In [6]:
# Complaint risk analysis

complaint_summary = (
    df.groupby("NumComplaints")
    .agg(
        total_customers=("CustomerId", "count"),
        churned_customers=("Churn Flag", "sum"),
        churn_rate_percentage=("Churn Flag", lambda x: round(x.mean() * 100, 2))
    )
    .reset_index()
)

print(complaint_summary)

    NumComplaints  total_customers  churned_customers  churn_rate_percentage
0               0            10575                315                   2.98
1               1            10469                448                   4.28
2               2            10612                566                   5.33
3               3            10359                780                   7.53
4               4            10517               1018                   9.68
5               5            10746               1165                  10.84
6               6            10512               1443                  13.73
7               7            10564               1700                  16.09
8               8            10398               1967                  18.92
9               9            10409               2221                  21.34
10             10            10479               2471                  23.58


In [7]:
# Complaint risk groups

df["Complaint Risk Group"] = df["NumComplaints"].apply(
    lambda x: "Low Complaints" if x <= 2
    else "Medium Complaints" if x <= 6
    else "High Complaints"
)

complaint_group_summary = (
    df.groupby("Complaint Risk Group")
    .agg(
        total_customers=("CustomerId", "count"),
        churned_customers=("Churn Flag", "sum"),
        churn_rate_percentage=("Churn Flag", lambda x: round(x.mean() * 100, 2))
    )
    .reset_index()
    .sort_values("churn_rate_percentage", ascending=False)
)

print(complaint_group_summary)

  Complaint Risk Group  total_customers  churned_customers  \
0      High Complaints            41850               8359   
2    Medium Complaints            42134               4406   
1       Low Complaints            31656               1329   

   churn_rate_percentage  
0                  19.97  
2                  10.46  
1                   4.20  


In [10]:
# Credit score risk analysis

def credit_group(score):
    if score < 580:
        return "Poor Credit"
    elif score <= 669:
        return "Fair Credit"
    elif score <= 739:
        return "Good Credit"
    elif score <= 799:
        return "Very Good Credit"
    else:
        return "Excellent Credit"

df["Credit Score Group"] = df["Credit Score"].apply(credit_group)

credit_summary = (
    df.groupby("Credit Score Group")
    .agg(
        total_customers=("CustomerId", "count"),
        churned_customers=("Churn Flag", "sum"),
        churn_rate_percentage=("Churn Flag", lambda x: round(x.mean() * 100, 2)),
        avg_balance=("Balance", lambda x: round(x.mean(), 2)),
        avg_outstanding_loans=("Outstanding Loans", lambda x: round(x.mean(), 2))
    )
    .reset_index()
    .sort_values("churn_rate_percentage", ascending=False)
)

print(credit_summary)
print(credit_summary.to_string(index=False))

  Credit Score Group  total_customers  churned_customers  \
3        Poor Credit            58937              10175   
1        Fair Credit            18863               1811   
2        Good Credit            14659               1052   
4   Very Good Credit            12679                653   
0   Excellent Credit            10502                403   

   churn_rate_percentage  avg_balance  avg_outstanding_loans  
3                  17.26    124429.01               25507.60  
1                   9.60    125245.24               25570.49  
2                   7.18    124276.29               25604.34  
4                   5.15    124587.41               25212.30  
0                   3.84    125299.43               25562.53  
Credit Score Group  total_customers  churned_customers  churn_rate_percentage  avg_balance  avg_outstanding_loans
       Poor Credit            58937              10175                  17.26    124429.01               25507.60
       Fair Credit            188

In [11]:
# Product engagement analysis

product_summary = (
    df.groupby("NumOfProducts")
    .agg(
        total_customers=("CustomerId", "count"),
        churned_customers=("Churn Flag", "sum"),
        churn_rate_percentage=("Churn Flag", lambda x: round(x.mean() * 100, 2)),
        avg_complaints=("NumComplaints", lambda x: round(x.mean(), 2)),
        avg_credit_score=("Credit Score", lambda x: round(x.mean(), 2)),
        avg_balance=("Balance", lambda x: round(x.mean(), 2))
    )
    .reset_index()
    .sort_values("NumOfProducts")
)

print(product_summary.to_string(index=False))

 NumOfProducts  total_customers  churned_customers  churn_rate_percentage  avg_complaints  avg_credit_score  avg_balance
             1            22898               4778                  20.87            5.01            574.52    125151.25
             2            23451               3851                  16.42            4.96            574.64    124700.77
             3            23198               2644                  11.40            4.99            574.33    124562.81
             4            23023               1807                   7.85            5.02            574.70    123875.96
             5            23070               1014                   4.40            4.97            573.30    124906.93


In [12]:
# Combined churn risk profile

df["Complaint Group"] = df["NumComplaints"].apply(
    lambda x: "High Complaints" if x >= 7 else "Low/Medium Complaints"
)

df["Credit Group"] = df["Credit Score"].apply(
    lambda x: "Poor Credit" if x < 580 else "Non-Poor Credit"
)

df["Product Group"] = df["NumOfProducts"].apply(
    lambda x: "Low Product Engagement" if x <= 2 else "High Product Engagement"
)

risk_profile_summary = (
    df.groupby(["Complaint Group", "Credit Group", "Product Group"])
    .agg(
        total_customers=("CustomerId", "count"),
        churned_customers=("Churn Flag", "sum"),
        churn_rate_percentage=("Churn Flag", lambda x: round(x.mean() * 100, 2))
    )
    .reset_index()
    .sort_values("churn_rate_percentage", ascending=False)
)

print(risk_profile_summary.to_string(index=False))

      Complaint Group    Credit Group           Product Group  total_customers  churned_customers  churn_rate_percentage
      High Complaints     Poor Credit  Low Product Engagement             8502               2941                  34.59
      High Complaints Non-Poor Credit  Low Product Engagement             8232               1726                  20.97
      High Complaints     Poor Credit High Product Engagement            12785               2678                  20.95
Low/Medium Complaints     Poor Credit  Low Product Engagement            15102               2990                  19.80
      High Complaints Non-Poor Credit High Product Engagement            12331               1014                   8.22
Low/Medium Complaints     Poor Credit High Product Engagement            22548               1566                   6.95
Low/Medium Complaints Non-Poor Credit  Low Product Engagement            14513                972                   6.70
Low/Medium Complaints Non-Poor C

In [13]:
# Churn risk scoring system

df["Risk Score"] = (
    (df["NumComplaints"] >= 7).astype(int)
    + (df["Credit Score"] < 580).astype(int)
    + (df["NumOfProducts"] <= 2).astype(int)
)

def risk_level(score):
    if score == 3:
        return "High Risk"
    elif score == 2:
        return "Medium Risk"
    elif score == 1:
        return "Low Risk"
    else:
        return "Very Low Risk"

df["Risk Level"] = df["Risk Score"].apply(risk_level)

risk_tier_summary = (
    df.groupby(["Risk Score", "Risk Level"])
    .agg(
        total_customers=("CustomerId", "count"),
        churned_customers=("Churn Flag", "sum"),
        churn_rate_percentage=("Churn Flag", lambda x: round(x.mean() * 100, 2))
    )
    .reset_index()
    .sort_values("Risk Score", ascending=False)
)

print(risk_tier_summary.to_string(index=False))

 Risk Score    Risk Level  total_customers  churned_customers  churn_rate_percentage
          3     High Risk             8502               2941                  34.59
          2   Medium Risk            36119               7394                  20.47
          1      Low Risk            49392               3552                   7.19
          0 Very Low Risk            21627                207                   0.96


In [14]:
# Save final summary outputs

complaint_group_summary.to_csv("complaint_group_summary.csv", index=False)
credit_summary.to_csv("credit_score_summary.csv", index=False)
product_summary.to_csv("product_engagement_summary.csv", index=False)
risk_profile_summary.to_csv("risk_profile_summary.csv", index=False)
risk_tier_summary.to_csv("risk_tier_summary.csv", index=False)

print("Python analysis completed successfully.")

Python analysis completed successfully.
